# Purchase Prediction — Google Merchandise Store (V3)
**Author:** Jason Valade  
**Dataset:** `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*` (Nov 2020 – Jan 2021)  
**Notebook:** `notebooks/03_purchase_prediction.ipynb`

---

## 1  Objective and Prediction Design

| Design dimension | Choice |
|---|---|
| **Unit of observation** | One product-view session — a GA4 session containing at least one `view_item` event |
| **Prediction moment** | Immediately after the session's **first** `view_item` event |
| **Target** | `purchased_later_in_session = 1` if a `purchase` event occurs strictly after that first `view_item` in the same session; 0 otherwise |
| **Feature scope** | Only information available at or before the prediction moment: device, geography, acquisition channel, item details on the viewed product, and behavioural events that occurred before the first product view |
| **Excluded signals** | `add_to_cart`, `begin_checkout`, purchase counts, transaction IDs, revenue, ecommerce totals, and any post-`view_item` behaviour |
| **Attribution caveat** | `acquisition_source` and `acquisition_medium` reflect **first-user acquisition** (how the user was originally brought to the store), not session-level attribution |
| **Causal scope** | This model estimates **propensity and association**, not causal effects. Differences in predicted purchase probability between segments do not establish that any feature *causes* higher conversion. Improving conversion requires controlled experiments. |

**Why this design prevents leakage:**  
Every event-count feature is bounded by `event_timestamp < first_view_item_timestamp` (strict less-than).  
Item details come from the `view_item` event itself — the prediction moment — not from any subsequent event.  
`add_to_cart`, `begin_checkout`, and `purchase` are absent from all feature aggregations.

## 2  Load and Validate

In [1]:
import warnings; warnings.filterwarnings("ignore")
import random, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

import sklearn, scipy
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Python   {sys.version.split()[0]}")
print(f"pandas   {pd.__version__}")
print(f"numpy    {np.__version__}")
print(f"sklearn  {sklearn.__version__}")
print(f"scipy    {scipy.__version__}")
print(f"matplotlib {matplotlib.__version__}")


Python   3.13.3
pandas   3.0.3
numpy    2.5.1
sklearn  1.9.0
scipy    1.18.0
matplotlib 3.11.1


In [2]:
# ── Paths: resolves from repo root or notebooks/ directory ───────────────────
_here = Path.cwd()
REPO = _here.parent if _here.name == "notebooks" else _here
DATA = REPO / "data/processed/demo/model_features.csv.gz"
IMG  = REPO / "images"
IMG.mkdir(exist_ok=True)

df = pd.read_csv(DATA, parse_dates=["session_date"])
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


Loaded: 77,020 rows × 20 columns


,session_id,session_date,first_view_item_timestamp,device_category,country,acquisition_source,acquisition_medium,first_item_name,first_item_category,first_item_price,hour_of_day,day_of_week,seconds_from_session_start_to_first_view,page_views_before_first_view,scroll_events_before_first_view,search_events_before_first_view,promotion_views_before_first_view,engagement_events_before_first_view,is_new_visitor,purchased_later_in_session
0,10054495.7553807982_4426974777,2020-11-01,1604241472072520,desktop,Canada,google,organic,google speckled beanie navy,Home/Apparel/Hats/,20.0,14,1,15.555770,1,0,0,1,1,1,0
1,1013442.5000387623_1129172264,2020-11-01,1604202756756097,mobile,United States,(direct),(none),google speckled beanie navy,Home/Apparel/Hats/,20.0,3,1,13.004764,1,0,0,1,1,1,0
2,10254930.7336842463_7802560767,2020-11-01,1604236063167935,desktop,Belgium,google,organic,google lapel pin,NaN,8.0,13,1,290.543881,8,4,2,0,6,1,0


In [3]:
# ── Assertion-level validation ────────────────────────────────────────────────
assert df.shape == (77020, 20), f"Unexpected shape: {df.shape}"
assert df["session_id"].nunique() == len(df), "Duplicate session_ids found"
assert set(df["purchased_later_in_session"].unique()) <= {0, 1}, "Target contains values outside {0,1}"

vc = df["purchased_later_in_session"].value_counts()
assert vc[1] == 4688,  f"Expected 4,688 positives; got {vc[1]}"
assert vc[0] == 72332, f"Expected 72,332 negatives; got {vc[0]}"
assert df["session_date"].min().date().isoformat() == "2020-11-01", "Unexpected min date"
assert df["session_date"].max().date().isoformat() == "2021-01-31", "Unexpected max date"
assert (df["seconds_from_session_start_to_first_view"] < 0).sum() == 0, "Negative seconds found"

print("\u2713 All assertions passed")
print()

# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].rename("SQL NULL count")
print("Missing-value summary (SQL NULLs):")
print(missing.to_frame())
print()

# ── Class balance ─────────────────────────────────────────────────────────────
print("Class balance:")
balance = vc.rename("count").to_frame()
balance["pct"] = (balance["count"] / len(df) * 100).round(2)
print(balance)
print()
print(f"Purchase rate: {vc[1]/len(df)*100:.2f}%  — imbalanced; PR-AUC used as primary metric")


✓ All assertions passed

Missing-value summary (SQL NULLs):
                     SQL NULL count
first_item_category            2187
first_item_price              20697

Class balance:
                            count    pct
purchased_later_in_session              
0                           72332  93.91
1                            4688   6.09

Purchase rate: 6.09%  — imbalanced; PR-AUC used as primary metric


## 3  Chronological Split

Sessions are sorted by `session_date` and divided into three non-overlapping windows.  
No shuffling is used. `session_id`, `session_date`, and `first_view_item_timestamp` are  
excluded from model features.

| Split | Window |
|---|---|
| **Train** | 2020-11-01 → 2020-12-31 |
| **Validation** | 2021-01-01 → 2021-01-15 |
| **Test** | 2021-01-16 → 2021-01-31 |

Validation is used for threshold selection only. Test data is never used during fitting or  
threshold optimisation.

In [4]:
train = df[df["session_date"] <= "2020-12-31"].copy()
val   = df[(df["session_date"] >= "2021-01-01") & (df["session_date"] <= "2021-01-15")].copy()
test  = df[df["session_date"] >= "2021-01-16"].copy()

# Confirm no overlap
assert len(set(train["session_id"]) & set(val["session_id"]))  == 0
assert len(set(val["session_id"])   & set(test["session_id"])) == 0
assert len(train) + len(val) + len(test) == len(df)

split_summary = []
for name, s in [("train", train), ("val", val), ("test", test)]:
    split_summary.append({
        "split":         name,
        "start_date":    s["session_date"].min().date().isoformat(),
        "end_date":      s["session_date"].max().date().isoformat(),
        "rows":          len(s),
        "purchases":     int(s["purchased_later_in_session"].sum()),
        "purchase_rate": f"{s['purchased_later_in_session'].mean()*100:.2f}%",
    })

print(pd.DataFrame(split_summary).to_string(index=False))


split start_date   end_date  rows  purchases purchase_rate
train 2020-11-01 2020-12-31 53917       3618         6.71%
  val 2021-01-01 2021-01-15  9445        382         4.04%
 test 2021-01-16 2021-01-31 13658        688         5.04%


## 4  Deterministic Feature Engineering

All transformations below are **deterministic** (no statistics are computed).  
They are applied identically to train, validation, and test without fitting.

| Transformation | Rationale |
|---|---|
| `item_metadata_missing` | Binary flag when price is null or item name/category is unknown. Preserves missingness signal for the model. |
| `long_pre_view_session` | Binary flag for sessions where the visitor browsed > 30 min before the first product view. |
| Cap seconds at 1 800 | Prevents outliers from dominating a linear model. 30 min is a natural session-engagement ceiling. |
| `log1p` on capped seconds | Compresses the right-skewed distribution. |
| `log1p` on count features | Count features are zero-inflated and right-skewed. |
| Cyclic encoding of `hour_of_day` (period 24) | Encodes the circular nature of time-of-day (23:00 and 00:00 are adjacent). |
| Cyclic encoding of `day_of_week` (period 7) | BigQuery DAYOFWEEK: 1=Sun…7=Sat. Shifted to 0-indexed before encoding. |

**Not included:** `add_to_cart`, `begin_checkout`, purchase timing, revenue, transactions, or any post-view behaviour.

In [5]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    df2 = frame.copy()

    # ── Binary flags ─────────────────────────────────────────────────────────
    df2["item_metadata_missing"] = (
        df2["first_item_price"].isna()
        | (df2["first_item_name"] == "(unknown)")
        | df2["first_item_category"].isna()
        | (df2["first_item_category"] == "(unknown)")
    ).astype(int)

    df2["long_pre_view_session"] = (
        df2["seconds_from_session_start_to_first_view"] > 1800
    ).astype(int)

    # ── Cap + log1p seconds ──────────────────────────────────────────────────
    df2["seconds_capped"] = df2["seconds_from_session_start_to_first_view"].clip(upper=1800)
    df2["seconds_log1p"]  = np.log1p(df2["seconds_capped"])

    # ── log1p count features ─────────────────────────────────────────────────
    for col in [
        "page_views_before_first_view",
        "scroll_events_before_first_view",
        "search_events_before_first_view",
        "promotion_views_before_first_view",
        "engagement_events_before_first_view",
    ]:
        df2[col + "_log1p"] = np.log1p(df2[col])

    # ── Cyclic encoding: hour_of_day (period = 24) ────────────────────────────
    df2["hour_sin"] = np.sin(2 * np.pi * df2["hour_of_day"] / 24)
    df2["hour_cos"] = np.cos(2 * np.pi * df2["hour_of_day"] / 24)

    # ── Cyclic encoding: day_of_week (period = 7; 0-indexed) ─────────────────
    # BigQuery DAYOFWEEK: 1=Sun, 2=Mon, ..., 7=Sat -> shift to 0-indexed
    df2["dow_sin"] = np.sin(2 * np.pi * (df2["day_of_week"] - 1) / 7)
    df2["dow_cos"] = np.cos(2 * np.pi * (df2["day_of_week"] - 1) / 7)

    return df2

train = engineer_features(train)
val   = engineer_features(val)
test  = engineer_features(test)

new_cols = [
    "item_metadata_missing", "long_pre_view_session",
    "seconds_log1p", "page_views_before_first_view_log1p",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]
print("Engineered feature sample (train):")
print(train[new_cols].describe().round(3))
print()
print(f"item_metadata_missing sessions : {train['item_metadata_missing'].sum():,}")
print(f"long_pre_view_session sessions : {train['long_pre_view_session'].sum():,}")


Engineered feature sample (train):
       item_metadata_missing  long_pre_view_session  seconds_log1p  \
count              53917.000              53917.000      53917.000   
mean                   0.268                  0.007          3.714   
std                    0.443                  0.081          1.570   
min                    0.000                  0.000          0.000   
25%                    0.000                  0.000          2.525   
50%                    0.000                  0.000          3.770   
75%                    1.000                  0.000          4.774   
max                    1.000                  1.000          7.496   

       page_views_before_first_view_log1p   hour_sin   hour_cos    dow_sin  \
count                           53917.000  53917.000  53917.000  53917.000   
mean                                1.142      0.002      0.002      0.077   
std                                 0.622      0.709      0.705      0.717   
min                   

## 5  Preprocessing Pipeline

All imputers, encoders, and scalers are fitted **on training data only** and applied  
identically to validation and test via `transform`.

| Feature group | Steps |
|---|---|
| **Categorical** (6) | `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=50)` |
| **Price** (1) | `SimpleImputer(median)` → `log1p` → `StandardScaler` *(LR)* or `log1p` only *(RF)* |
| **Other numerics** (13) | `StandardScaler` *(LR)* or `passthrough` *(RF)* |

`min_frequency=50` collapses rare categories (< 50 train occurrences) into an  
`infrequent` bucket. This keeps the feature matrix tractable for the 416-level  
`first_item_name` and 109-level `country` columns while preserving signal from  
high-frequency values.

In [6]:
# ── Feature lists ─────────────────────────────────────────────────────────────
CAT_FEATURES = [
    "device_category", "country", "acquisition_source",
    "acquisition_medium", "first_item_name", "first_item_category",
]
PRICE_FEATURE = ["first_item_price"]
NUM_FEATURES  = [
    "seconds_log1p",
    "page_views_before_first_view_log1p",
    "scroll_events_before_first_view_log1p",
    "search_events_before_first_view_log1p",
    "promotion_views_before_first_view_log1p",
    "engagement_events_before_first_view_log1p",
    "hour_sin", "hour_cos",
    "dow_sin",  "dow_cos",
    "item_metadata_missing",
    "long_pre_view_session",
    "is_new_visitor",
]
MODEL_FEATURES = CAT_FEATURES + PRICE_FEATURE + NUM_FEATURES
TARGET = "purchased_later_in_session"

X_train = train[MODEL_FEATURES]
y_train = train[TARGET]
X_val   = val  [MODEL_FEATURES]
y_val   = val  [TARGET]
X_test  = test [MODEL_FEATURES]
y_test  = test [TARGET]

print(f"X_train: {X_train.shape} | y_train positives: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"X_val  : {X_val.shape}   | y_val   positives: {y_val.sum()} ({y_val.mean()*100:.2f}%)")
print(f"X_test : {X_test.shape}  | y_test  positives: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

# ── Preprocessing sub-pipelines ────────────────────────────────────────────────
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50,
        sparse_output=False,
    )),
])

price_pipe_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p",   FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler",  StandardScaler()),
])
price_pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p",   FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
])

preprocessor_lr = ColumnTransformer([
    ("cat",   cat_pipe,       CAT_FEATURES),
    ("price", price_pipe_lr,  PRICE_FEATURE),
    ("num",   Pipeline([("scaler", StandardScaler())]),  NUM_FEATURES),
], remainder="drop", verbose_feature_names_out=True)

preprocessor_rf = ColumnTransformer([
    ("cat",   cat_pipe,       CAT_FEATURES),
    ("price", price_pipe_rf,  PRICE_FEATURE),
    ("num",   "passthrough",  NUM_FEATURES),
], remainder="drop", verbose_feature_names_out=True)

print("\nPreprocessing pipelines defined.")
print(f"Total model features going in: {len(MODEL_FEATURES)}")


X_train: (53917, 20) | y_train positives: 3618 (6.71%)
X_val  : (9445, 20)   | y_val   positives: 382 (4.04%)
X_test : (13658, 20)  | y_test  positives: 688 (5.04%)

Preprocessing pipelines defined.
Total model features going in: 20


## 6  Models

Three models are trained on the training split only:

| Model | Role |
|---|---|
| `DummyClassifier(strategy='prior')` | No-skill benchmark — always predicts the training purchase rate |
| `LogisticRegression(class_weight='balanced')` | Interpretable linear baseline |
| `RandomForestClassifier(class_weight='balanced_subsample')` | Nonlinear tree-based comparison model |

`class_weight='balanced'` / `balanced_subsample` up-weights the minority class  
(purchasers) to compensate for the ~6.7% training purchase rate.

In [7]:
# ── Instantiate ───────────────────────────────────────────────────────────────
dummy = DummyClassifier(strategy="prior", random_state=SEED)

lr_pipe = Pipeline([
    ("prep", preprocessor_lr),
    ("clf",  LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

rf_pipe = Pipeline([
    ("prep", preprocessor_rf),
    ("clf",  RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
    )),
])

# ── Fit (training data only) ──────────────────────────────────────────────────
print("Fitting DummyClassifier...")
dummy.fit(X_train, y_train)
print("  done.")

print("Fitting LogisticRegression...")
lr_pipe.fit(X_train, y_train)
print("  done.")

print("Fitting RandomForestClassifier (n_estimators=300, this may take ~30 s)...")
rf_pipe.fit(X_train, y_train)
print("  done.")

print("\nAll three models fitted on training data only.")


Fitting DummyClassifier...
  done.
Fitting LogisticRegression...


  done.
Fitting RandomForestClassifier (n_estimators=300, this may take ~30 s)...


  done.

All three models fitted on training data only.


## 7  Validation Evaluation

**Primary metric:** PR-AUC (`average_precision_score`)  
Preferred over ROC-AUC and accuracy for imbalanced targets because it focuses on  
the minority class (purchasers) and is not inflated by the large number of  
true negatives.

**No-skill baseline:** PR-AUC ≈ validation prevalence (~4.0%)

**Secondary metrics:** ROC-AUC, Brier score, precision, recall, F1, top-decile lift,  
top-decile purchase capture rate.

**Threshold selection:** For LR and RF, the classification threshold is chosen on the  
validation set by maximising F1. Uncalibrated thresholds are stored here for  
reference; the calibrated threshold (Section 8b) is used for final test evaluation.

In [8]:
# ── Helper functions ──────────────────────────────────────────────────────────
def predict_proba_col(model, X):
    return model.predict_proba(X)[:, 1]

def compute_metrics(y_true, proba, threshold):
    preds    = (proba >= threshold).astype(int)
    n        = len(y_true)
    decile_n = max(1, n // 10)
    order    = np.argsort(proba)[::-1]
    y_sorted = np.array(y_true)[order]
    top_purch = y_sorted[:decile_n].sum()
    top_rate  = top_purch / decile_n
    base_rate = np.mean(y_true)
    lift      = top_rate / base_rate if base_rate > 0 else np.nan
    capture   = top_purch / y_true.sum() if y_true.sum() > 0 else np.nan
    return {
        "pr_auc":    average_precision_score(y_true, proba),
        "roc_auc":   roc_auc_score(y_true, proba),
        "brier":     brier_score_loss(y_true, proba),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall":    recall_score(y_true, preds, zero_division=0),
        "f1":        f1_score(y_true, preds, zero_division=0),
        "lift":      lift,
        "capture":   capture,
        "cm":        confusion_matrix(y_true, preds),
    }

def best_f1_threshold(proba, y_true, n_steps=199):
    ts  = np.linspace(0.01, 0.99, n_steps)
    f1s = [f1_score(y_true, (proba >= t).astype(int), zero_division=0) for t in ts]
    return float(ts[int(np.argmax(f1s))])

# ── Dummy val ─────────────────────────────────────────────────────────────────
dummy_proba_val   = predict_proba_col(dummy, X_val)
dummy_val_metrics = compute_metrics(y_val.values, dummy_proba_val, threshold=0.5)
dummy_thresh      = 0.5

# ── LR: uncalibrated threshold on validation ──────────────────────────────────
lr_proba_val  = predict_proba_col(lr_pipe, X_val)
lr_thresh     = best_f1_threshold(lr_proba_val, y_val.values)
lr_val_metrics = compute_metrics(y_val.values, lr_proba_val, threshold=lr_thresh)
print(f"LR  best-F1 threshold (validation, uncalibrated): {lr_thresh:.4f}")

# ── RF: uncalibrated threshold on validation ──────────────────────────────────
rf_proba_val  = predict_proba_col(rf_pipe, X_val)
rf_thresh     = best_f1_threshold(rf_proba_val, y_val.values)
rf_val_metrics = compute_metrics(y_val.values, rf_proba_val, threshold=rf_thresh)
print(f"RF  best-F1 threshold (validation, uncalibrated): {rf_thresh:.4f}")


LR  best-F1 threshold (validation, uncalibrated): 0.7227


RF  best-F1 threshold (validation, uncalibrated): 0.6138


In [9]:
# ── Validation comparison table (uncalibrated) ────────────────────────────────
def fmt_row(name, m, thresh):
    return {
        "Model":         name,
        "PR-AUC":        round(m["pr_auc"],    4),
        "ROC-AUC":       round(m["roc_auc"],   4),
        "Brier":         round(m["brier"],      4),
        "Precision":     round(m["precision"],  4),
        "Recall":        round(m["recall"],     4),
        "F1":            round(m["f1"],         4),
        "Lift@Decile1":  round(m["lift"],       3),
        "Capture@D1%":   f"{m['capture']*100:.1f}%" if not np.isnan(m["capture"]) else "—",
        "Threshold":     round(thresh,          4),
    }

val_table = pd.DataFrame([
    fmt_row("DummyClassifier",    dummy_val_metrics, dummy_thresh),
    fmt_row("LogisticRegression", lr_val_metrics,    lr_thresh),
    fmt_row("RandomForest",       rf_val_metrics,    rf_thresh),
])
print("=== Validation Comparison Table (uncalibrated probabilities) ===")
print()
print(val_table.to_string(index=False))
print()
print(f"No-skill PR-AUC baseline (= val prevalence): {y_val.mean():.4f}")


=== Validation Comparison Table (uncalibrated probabilities) ===

             Model  PR-AUC  ROC-AUC  Brier  Precision  Recall     F1  Lift@Decile1 Capture@D1%  Threshold
   DummyClassifier  0.0404   0.5000 0.0395     0.0000  0.0000 0.0000         0.760        7.6%     0.5000
LogisticRegression  0.0980   0.7490 0.1836     0.1319  0.2749 0.1783         3.064       30.6%     0.7227
      RandomForest  0.0997   0.7524 0.2018     0.1168  0.3377 0.1736         2.855       28.5%     0.6138

No-skill PR-AUC baseline (= val prevalence): 0.0404


## 8  Model Selection, Calibration, and Final Test Evaluation

### 8a  Model selection

The winner is selected on **validation PR-AUC only** — identical rule to the  
uncalibrated comparison above. RandomForest is selected before calibration is applied.  
The test set plays no role in model selection.

### 8b  Sigmoid calibration

`class_weight='balanced_subsample'` causes RandomForest to output scores  
weighted toward the minority class; these scores are well-ordered but  
systematically mis-scaled as probability estimates. Sigmoid calibration maps  
the raw scores to a more realistic probability scale without changing the  
model's rank ordering.

**Why class weighting distorts probabilities:**  
Balanced class weighting inflates predicted scores for the minority class so  
that the classifier treats it as if it were more prevalent than it actually is.  
The raw probabilities therefore tend to be too high relative to the true  
population rate (~5–7%). Sigmoid calibration fits a monotonic function that  
maps these inflated scores back toward the observed positive rate.

**Calibration methodology — no test data involved at any step:**
- The underlying RandomForest pipeline was already fitted on training data only.
- `FrozenEstimator` (sklearn ≥ 1.4) wraps the fitted pipeline, preventing any  
  weight updates during calibration.
- `CalibratedClassifierCV(method='sigmoid', cv=None)` is fitted on  
  `(X_val, y_val)` only. `cv=None` combined with `FrozenEstimator` is the  
  sklearn 1.4+ replacement for the removed `cv='prefit'` parameter.
- Validation data is used **solely** for calibration fitting and threshold  
  re-selection.
- **Test data remained excluded from fitting, model selection, calibration,  
  and threshold selection throughout.**

### 8c  Final test evaluation

The calibrated pipeline and calibrated threshold are evaluated on the untouched  
test set exactly once. Test accuracy alone is not reported as a success criterion  
because accuracy is misleading on an imbalanced target (~5% positive rate).

In [10]:
# ── Select winner by validation PR-AUC (uncalibrated) ───────────────────────
if rf_val_metrics["pr_auc"] >= lr_val_metrics["pr_auc"]:
    WINNER_NAME   = "RandomForest"
    winner_pipe   = rf_pipe
else:
    WINNER_NAME   = "LogisticRegression"
    winner_pipe   = lr_pipe

print(f"Winner (validation PR-AUC): {WINNER_NAME}")
print(f"  LR  val PR-AUC (uncalibrated): {lr_val_metrics['pr_auc']:.4f}")
print(f"  RF  val PR-AUC (uncalibrated): {rf_val_metrics['pr_auc']:.4f}")
print()
print("Proceeding to sigmoid calibration on validation data.")


Winner (validation PR-AUC): RandomForest
  LR  val PR-AUC (uncalibrated): 0.0980
  RF  val PR-AUC (uncalibrated): 0.0997

Proceeding to sigmoid calibration on validation data.


In [11]:
# ── Sigmoid calibration — fitted on validation data only ─────────────────────
# FrozenEstimator prevents CalibratedClassifierCV from refitting the underlying
# model. cv=None with FrozenEstimator is the sklearn >= 1.4 API;
# cv='prefit' was removed in sklearn 1.4.
_frozen_winner   = FrozenEstimator(winner_pipe)
calibrated_model = CalibratedClassifierCV(_frozen_winner, method="sigmoid", cv=None)
calibrated_model.fit(X_val, y_val)

print(f"Sigmoid calibrator fitted on validation data ({len(y_val):,} rows).")
print(f"Underlying {WINNER_NAME} pipeline: FROZEN — weights unchanged.")
print()

# ── Calibrated validation probabilities ──────────────────────────────────────
cal_proba_val   = calibrated_model.predict_proba(X_val)[:, 1]

# ── Brier score comparison on validation ─────────────────────────────────────
uncal_brier_val = brier_score_loss(y_val, rf_proba_val)
cal_brier_val   = brier_score_loss(y_val, cal_proba_val)
print(f"Validation Brier — uncalibrated RF : {uncal_brier_val:.4f}")
print(f"Validation Brier — calibrated RF   : {cal_brier_val:.4f}")
if cal_brier_val < uncal_brier_val:
    print("  ✓ Calibrated Brier is lower on validation (better probability reliability).")
else:
    print("  ⚠ Calibrated Brier is not lower on validation.")
print()

# ── Re-select F1 threshold on calibrated validation probabilities ─────────────
cal_thresh     = best_f1_threshold(cal_proba_val, y_val.values)
cal_val_metrics = compute_metrics(y_val.values, cal_proba_val, threshold=cal_thresh)
print(f"Calibrated threshold (max-F1 on calibrated val proba) : {cal_thresh:.4f}")
print(f"Calibrated val PR-AUC  : {cal_val_metrics['pr_auc']:.4f}")
print(f"Calibrated val F1      : {cal_val_metrics['f1']:.4f}")
print()
print("Calibrated threshold locked. No further threshold changes.")


Sigmoid calibrator fitted on validation data (9,445 rows).
Underlying RandomForest pipeline: FROZEN — weights unchanged.



Validation Brier — uncalibrated RF : 0.2018
Validation Brier — calibrated RF   : 0.0375
  ✓ Calibrated Brier is lower on validation (better probability reliability).



Calibrated threshold (max-F1 on calibrated val proba) : 0.0892
Calibrated val PR-AUC  : 0.0997
Calibrated val F1      : 0.1739

Calibrated threshold locked. No further threshold changes.


In [12]:
# ── Final test evaluation — calibrated model + locked calibrated threshold ────
# One evaluation on untouched test data. No further adjustments after this cell.
cal_proba_test   = calibrated_model.predict_proba(X_test)[:, 1]
cal_test_metrics = compute_metrics(y_test.values, cal_proba_test, threshold=cal_thresh)

# Uncalibrated test proba used only for Brier comparison
uncal_proba_test = predict_proba_col(winner_pipe, X_test)
uncal_brier_test = brier_score_loss(y_test, uncal_proba_test)
cal_brier_test   = cal_test_metrics["brier"]

test_prev      = y_test.mean()
noskill_pr_auc = test_prev

print("=" * 60)
print(f"  FINAL TEST RESULTS — {WINNER_NAME} (Sigmoid Calibrated)")
print("=" * 60)
print(f"  Test sessions              : {len(y_test):,}")
print(f"  Test prevalence            : {test_prev*100:.2f}%  ({y_test.sum()} purchases)")
print(f"  No-skill PR-AUC            : {noskill_pr_auc:.4f}  (= prevalence)")
print(f"  PR-AUC                     : {cal_test_metrics['pr_auc']:.4f}")
print(f"  ROC-AUC                    : {cal_test_metrics['roc_auc']:.4f}")
print(f"  Brier — uncalibrated       : {uncal_brier_test:.4f}")
brier_note = "  ✓ improved" if cal_brier_test < uncal_brier_test else "  (not improved on test set)"
print(f"  Brier — calibrated         : {cal_brier_test:.4f}{brier_note}")
print(f"  Precision (thr={cal_thresh:.2f})      : {cal_test_metrics['precision']:.4f}")
print(f"  Recall    (thr={cal_thresh:.2f})      : {cal_test_metrics['recall']:.4f}")
print(f"  F1        (thr={cal_thresh:.2f})      : {cal_test_metrics['f1']:.4f}")
print(f"  Top-decile lift            : {cal_test_metrics['lift']:.2f}x")
print(f"  Top-decile capture         : {cal_test_metrics['capture']*100:.1f}%")
print("=" * 60)
print()
print("Confusion matrix (rows=actual, cols=predicted):")
cm = cal_test_metrics["cm"]
print(pd.DataFrame(cm, index=["Actual 0","Actual 1"], columns=["Pred 0","Pred 1"]))


  FINAL TEST RESULTS — RandomForest (Sigmoid Calibrated)
  Test sessions              : 13,658
  Test prevalence            : 5.04%  (688 purchases)
  No-skill PR-AUC            : 0.0504  (= prevalence)
  PR-AUC                     : 0.1402
  ROC-AUC                    : 0.7876
  Brier — uncalibrated       : 0.1778
  Brier — calibrated         : 0.0457  ✓ improved
  Precision (thr=0.09)      : 0.1578
  Recall    (thr=0.09)      : 0.3183
  F1        (thr=0.09)      : 0.2110
  Top-decile lift            : 3.13x
  Top-decile capture         : 31.2%

Confusion matrix (rows=actual, cols=predicted):
          Pred 0  Pred 1
Actual 0   11801    1169
Actual 1     469     219


## 9  Visualizations

In [13]:
# ── Chart 1: Precision-Recall curves — validation model comparison ────────────
# This chart compares LR and RF using uncalibrated validation probabilities.
# Calibration changes the probability scale but not the rank ordering, so
# it does not alter the PR-AUC values and is not re-presented here as a new
# model-selection exercise.
fig, ax = plt.subplots(figsize=(7.5, 5.5))

val_prev = y_val.mean()
ax.axhline(val_prev, color="#999999", linestyle="--", lw=1.4,
           label=f"No-skill baseline  PR-AUC = {val_prev:.3f}")

_colors = {"LogisticRegression": "#2196F3", "RandomForest": "#FF9800"}
for name, proba in [("LogisticRegression", lr_proba_val), ("RandomForest", rf_proba_val)]:
    prec_c, rec_c, _ = precision_recall_curve(y_val, proba)
    ap = average_precision_score(y_val, proba)
    ax.plot(rec_c, prec_c, lw=2.0, color=_colors[name], label=f"{name}  PR-AUC = {ap:.3f}")

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves — Validation Set", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
ax.grid(alpha=0.3)

plt.tight_layout()
_pr_path = IMG / "model_precision_recall.png"
plt.savefig(_pr_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {_pr_path.relative_to(REPO)}")


Saved: images/model_precision_recall.png


In [14]:
# ── Chart 2: Calibration curves — uncalibrated vs sigmoid calibrated ──────────
# Assessed on the untouched test set. strategy="quantile" places bin edges at
# equal-frequency quantiles of predicted probability, ensuring each bin
# contains approximately the same number of samples. This gives reliable
# calibration estimates even when predicted probabilities are concentrated
# in a narrow range, which is typical after sigmoid calibration.
frac_pos_uncal, mean_pred_uncal = calibration_curve(
    y_test, uncal_proba_test, n_bins=10, strategy="quantile")
frac_pos_cal, mean_pred_cal = calibration_curve(
    y_test, cal_proba_test,   n_bins=10, strategy="quantile")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle(
    f"Random Forest Probability Calibration \u2014 Test Set",
    fontsize=14, fontweight="bold", y=1.01,
)
fig.text(
    0.5, 0.97,
    "Equal-frequency bins; calibration fitted on validation data only",
    ha="center", fontsize=10, style="italic", color="#555555",
)

# ── Left panel: full 0-to-1 range ─────────────────────────────────────────────
ax0 = axes[0]
ax0.plot([0, 1], [0, 1], "--", color="#999999", lw=1.4, label="Perfect calibration")
ax0.plot(mean_pred_uncal, frac_pos_uncal, "s--", lw=1.8, color="#FF9800",
         label=f"Uncalibrated  Brier={uncal_brier_test:.4f}")
ax0.plot(mean_pred_cal,   frac_pos_cal,   "o-",  lw=2.0, color="#2196F3",
         label=f"Sigmoid calibrated  Brier={cal_brier_test:.4f}")
ax0.set_xlabel("Mean predicted probability", fontsize=11)
ax0.set_ylabel("Observed fraction of positives", fontsize=11)
ax0.set_title("Full probability range", fontsize=12, fontweight="bold")
ax0.legend(fontsize=9, loc="upper left")
ax0.set_xlim([0, 1]); ax0.set_ylim([0, 1])
ax0.grid(alpha=0.3)

# ── Right panel: zoom on calibrated range 0-0.20 ──────────────────────────────
ax1 = axes[1]
ax1.plot([0, 0.20], [0, 0.20], "--", color="#999999", lw=1.4, label="Perfect calibration")
ax1.plot(mean_pred_cal, frac_pos_cal, "o-", lw=2.0, color="#2196F3",
         label=f"Sigmoid calibrated  Brier={cal_brier_test:.4f}")
ax1.set_xlabel("Mean predicted probability", fontsize=11)
ax1.set_ylabel("Observed fraction of positives", fontsize=11)
ax1.set_title("Calibrated probability range", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9, loc="upper left")
ax1.set_xlim([0, 0.20]); ax1.set_ylim([0, 0.20])
ax1.grid(alpha=0.3)

plt.tight_layout()
_cal_path = IMG / "model_calibration.png"
plt.savefig(_cal_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {_cal_path.relative_to(REPO)}")


Saved: images/model_calibration.png


In [15]:
# ── Chart 3: Purchase rate by predicted-risk decile (calibrated, test set) ─────
_decile_df = pd.DataFrame({"proba": cal_proba_test, "target": y_test.values})
_decile_df["decile"] = pd.qcut(
    _decile_df["proba"], q=10, labels=False, duplicates="drop")
_decile_df["decile"] = 10 - _decile_df["decile"]   # 1 = highest predicted risk

_d_stats = (
    _decile_df.groupby("decile")
    .agg(sessions=("target","count"), purchases=("target","sum"))
    .reset_index()
)
_d_stats["purchase_rate_pct"] = _d_stats["purchases"] / _d_stats["sessions"] * 100
overall_pct = y_test.mean() * 100

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(
    _d_stats["decile"], _d_stats["purchase_rate_pct"],
    color="#2196F3", edgecolor="white", linewidth=0.6, width=0.7,
)
ax.axhline(overall_pct, color="crimson", linestyle="--", lw=1.6,
           label=f"Overall test rate  {overall_pct:.2f}%")

for bar, rate in zip(bars, _d_stats["purchase_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{rate:.1f}%", ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Score Decile  (1 = Highest Predicted Risk)", fontsize=12)
ax.set_ylabel("Purchase Rate (%)", fontsize=12)
ax.set_title(
    f"Purchase Rate by Predicted-Risk Decile — Test Set\n"
    f"{WINNER_NAME} (Sigmoid Calibrated)",
    fontsize=13, fontweight="bold",
)
ax.set_xticks(_d_stats["decile"])
ax.legend(fontsize=10)
ax.set_ylim(0, _d_stats["purchase_rate_pct"].max() * 1.25)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
_decile_path = IMG / "model_decile_lift.png"
plt.savefig(_decile_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {_decile_path.relative_to(REPO)}")


Saved: images/model_decile_lift.png


## 10  Interpretation and Limitations

### Logistic Regression Coefficients (directional associations)

The logistic regression coefficients indicate the direction and relative magnitude  
of the linear association between each feature and purchase log-odds, *after  
controlling for the other features in the model*. They are **not causal estimates**.  
A positive coefficient means the feature is associated with higher purchase propensity;  
a negative coefficient means lower. Do not interpret them as "this feature causes purchase".

In [16]:
# ── Top LR coefficients ────────────────────────────────────────────────────────
lr_prep    = lr_pipe.named_steps["prep"]
lr_clf     = lr_pipe.named_steps["clf"]
feat_names = lr_prep.get_feature_names_out()
coef_series = pd.Series(lr_clf.coef_[0], index=feat_names)
top_coef = (
    coef_series
    .abs()
    .sort_values(ascending=False)
    .head(20)
    .index
)
top_coef_df = coef_series[top_coef].sort_values(ascending=False).rename("coefficient").to_frame()
top_coef_df["direction"] = top_coef_df["coefficient"].apply(
    lambda x: "\u2191 higher purchase propensity" if x > 0 else "\u2193 lower purchase propensity"
)
print("Top 20 LogisticRegression features by absolute coefficient magnitude")
print("(associations, not causal effects — observational data only)")
print()
print(top_coef_df.to_string())


Top 20 LogisticRegression features by absolute coefficient magnitude
(associations, not causal effects — observational data only)

                                                         coefficient                     direction
cat__first_item_category_Women's                            1.360869  ↑ higher purchase propensity
cat__first_item_category_Home/Campus Collection/            1.050395  ↑ higher purchase propensity
cat__first_item_category_Drinkware                          0.994280  ↑ higher purchase propensity
cat__first_item_category_Home/Apparel/Socks/                0.979085  ↑ higher purchase propensity
cat__first_item_name_google incognito flap pack             0.957236  ↑ higher purchase propensity
cat__first_item_category_Home/Stationery/Writing/           0.943734  ↑ higher purchase propensity
cat__first_item_name_supernatural paper backpack           -0.917427   ↓ lower purchase propensity
cat__first_item_name_google cambridge campus zip hoodie    -0.967689   ↓ lowe

### Known Limitations

**Temporal drift**  
Monthly purchase rates varied substantially across the analysis window:
- November 2020: **6.12%**
- December 2020: **7.26%** (holiday shopping peak)
- January 2021: **4.63%**

The train split (Nov–Dec) has a higher base rate (6.71%) than validation (4.04%) and test (5.04%).  
A model trained on a higher-converting period may overestimate probability for January sessions.  
The threshold and model should be **recalibrated** if deployed on a different season's data.

**Data limitations**
- Only three months of obfuscated sample data are available. Results may not generalise to a  
  full production property or different time periods.
- The GA4 dataset is **obfuscated** — user identities, product names, and transaction values are  
  altered. Feature importances and conversion rates reflect this sample, not the live store.
- `acquisition_source` and `acquisition_medium` are **first-user acquisition** fields. They reflect  
  how the user was originally brought to the store, not how they arrived for this specific session.  
  Interpreting them as session-level channel attribution is incorrect.

**Feature limitations**
- **20,697 sessions lack complete item metadata** (price or item name/category).  
  The `item_metadata_missing` flag captures this, but these sessions are predicted with incomplete signal.
- **Class imbalance**: ~6.7% positive rate in training. Both models use `class_weight='balanced'` or  
  `'balanced_subsample'`. The selected threshold is tuned on calibrated validation probabilities.

**Modelling scope**
- This model identifies **correlates** of purchase propensity, not causes.  
- Differences in predicted probability between audience segments (e.g., mobile vs. desktop, organic  
  vs. paid) do not establish that device or channel *drives* conversion.
- **Controlled experiments** (A/B tests) are required before attributing conversion changes to any  
  intervention.
- The threshold and model require **ongoing monitoring and recalibration** as traffic mix, seasonality,  
  and product catalogue change.

## 11  Reproducibility

In [17]:
# ── Package versions ──────────────────────────────────────────────────────────
import importlib
print("Package versions used in this notebook:")
print("-" * 40)
for pkg in ["pandas", "numpy", "sklearn", "scipy", "matplotlib"]:
    mod = importlib.import_module(pkg if pkg != "sklearn" else "sklearn")
    print(f"  {pkg:<14} {mod.__version__}")
print(f"  {'python':<14} {sys.version.split()[0]}")
print()
print("Random seeds: 42 (random, numpy, model random_state parameters)")
print("Data file   :", DATA.relative_to(REPO))
print()
print("Chart outputs:")
for p in [
    IMG / "model_precision_recall.png",
    IMG / "model_calibration.png",
    IMG / "model_decile_lift.png",
]:
    status = "\u2713 exists" if p.exists() else "\u2717 MISSING"
    print(f"  {status}  {p.relative_to(REPO)}")


Package versions used in this notebook:
----------------------------------------
  pandas         3.0.3
  numpy          2.5.1
  sklearn        1.9.0
  scipy          1.18.0
  matplotlib     3.11.1
  python         3.13.3

Random seeds: 42 (random, numpy, model random_state parameters)
Data file   : data/processed/demo/model_features.csv.gz

Chart outputs:
  ✓ exists  images/model_precision_recall.png
  ✓ exists  images/model_calibration.png
  ✓ exists  images/model_decile_lift.png


In [18]:
# ── Concise results summary (generated from computed metrics) ─────────────────
print("=" * 64)
print("  RESULTS SUMMARY — 03_purchase_prediction.ipynb (calibrated)")
print("=" * 64)
print()
print("  SPLIT SIZES")
for row in split_summary:
    print(f"    {row['split']:<8} {row['rows']:>6,} rows  "
          f"{row['purchases']:>4} purchases  {row['purchase_rate']}")
print()
print("  VALIDATION COMPARISON (PR-AUC, uncalibrated — model selection basis)")
for _, r in val_table.iterrows():
    print(f"    {r['Model']:<22} PR-AUC={r['PR-AUC']:.4f}  "
          f"ROC-AUC={r['ROC-AUC']:.4f}  F1={r['F1']:.4f}")
print()
print(f"  WINNER (val PR-AUC)      : {WINNER_NAME}")
print(f"  CALIBRATION              : sigmoid via FrozenEstimator, fitted on validation only")
print(f"  CALIBRATED THRESHOLD     : {cal_thresh:.4f}  (max-F1 on calibrated val proba)")
print()
print("  FINAL TEST METRICS (sigmoid calibrated)")
print(f"    Prevalence             : {test_prev*100:.2f}%")
print(f"    No-skill PR-AUC        : {noskill_pr_auc:.4f}")
print(f"    PR-AUC                 : {cal_test_metrics['pr_auc']:.4f}")
print(f"    ROC-AUC                : {cal_test_metrics['roc_auc']:.4f}")
print(f"    Brier — uncalibrated   : {uncal_brier_test:.4f}")
brier_note = "  ✓ improved" if cal_brier_test < uncal_brier_test else "  (not improved on test)"
print(f"    Brier — calibrated     : {cal_brier_test:.4f}{brier_note}")
print(f"    Precision              : {cal_test_metrics['precision']:.4f}")
print(f"    Recall                 : {cal_test_metrics['recall']:.4f}")
print(f"    F1                     : {cal_test_metrics['f1']:.4f}")
print(f"    Top-decile lift        : {cal_test_metrics['lift']:.2f}x")
print(f"    Top-decile capture     : {cal_test_metrics['capture']*100:.1f}%")
print()
print("  All load assertions: PASSED")
print("=" * 64)


  RESULTS SUMMARY — 03_purchase_prediction.ipynb (calibrated)

  SPLIT SIZES
    train    53,917 rows  3618 purchases  6.71%
    val       9,445 rows   382 purchases  4.04%
    test     13,658 rows   688 purchases  5.04%

  VALIDATION COMPARISON (PR-AUC, uncalibrated — model selection basis)
    DummyClassifier        PR-AUC=0.0404  ROC-AUC=0.5000  F1=0.0000
    LogisticRegression     PR-AUC=0.0980  ROC-AUC=0.7490  F1=0.1783
    RandomForest           PR-AUC=0.0997  ROC-AUC=0.7524  F1=0.1736

  WINNER (val PR-AUC)      : RandomForest
  CALIBRATION              : sigmoid via FrozenEstimator, fitted on validation only
  CALIBRATED THRESHOLD     : 0.0892  (max-F1 on calibrated val proba)

  FINAL TEST METRICS (sigmoid calibrated)
    Prevalence             : 5.04%
    No-skill PR-AUC        : 0.0504
    PR-AUC                 : 0.1402
    ROC-AUC                : 0.7876
    Brier — uncalibrated   : 0.1778
    Brier — calibrated     : 0.0457  ✓ improved
    Precision              : 0.1578
